# IAM OCR data-augmentation benchmark

**Synthetic data → independent TrOCR training → held-out real IAM test.**

Default Run All generates **25,000 matched samples for SpiS-GAN and HiGAN+**, then
trains separate TrOCR models on **5,000 real IAM + each synthetic dataset**, plus
a **5,000-real-only control**. Both generators receive the same train-only
references, target texts and per-sample seeds; target text differs from its own
reference. No test samples or test lexicon enter generation/training.

Use **T4 ×2** and enable Internet. Required GAN checkpoints download automatically
from the GitHub release, with SHA-256 verification. Its native64 output is
explicitly exported32. W&B uses your existing `WANDB_KEY` Secret or environment.
This is not frozen OCR scoring of generated images.

### Before Run All

1. Select **GPU T4 ×2** and enable **Internet** in Kaggle settings.
2. No checkpoint attachment is needed: selected GAN weights download automatically.
   CHECKPOINT_PATHS is only for an intentional manual override.
3. Optional: attach prior benchmark Output and set PREVIOUS_OUTPUT_DIR for resume.
4. Optional: enable the Kaggle Secret `WANDB_KEY` for online logging.
5. Run All. The defaults perform generation **and** downstream TrOCR training.

You need only this notebook. It installs its own runtime, downloads/links IAM,
fetches generator inference code and runs the benchmark queue. It never trains
GANs or their OCR/writer teachers.


## 1. Choose the benchmark run

This notebook is standalone. Do not run the GAN/pretraining controller first.
Keep both toggles true to generate both datasets and then train the OCR arms.
For a continuation set PREVIOUS_OUTPUT_DIR below to the previous saved output.
The GAN checkpoint mapping and saved TrOCR checkpoints are separate inputs.


In [ ]:
#configs
from pathlib import Path
import os, sys, json, subprocess, time, hashlib, urllib.request, tempfile

DATA_AUGMENT = True       # Generate/resume saved synthetic IAM datasets.
TRAIN_TROCR = True        # Then train/resume separate recognizers + real-only control.
MODELS = ['spis_gan', 'higanplus']  # Optional additional supported model: 'fw_gan'.
GPU_SLOTS = ['0', '1']    # One independent worker per T4, not distributed training.
TIME_LIMIT = 10.5         # Hours including setup; leave margin for Kaggle saving.
SEED = 123456            # Shared split, manifest and TrOCR seed across arms.
REAL_COUNT = 5000        # X real IAM training rows, same X for every arm.
SYNTHETIC_COUNT = 25000  # Per generator, at most 25k; no quality filtering.
COHORT_HEIGHT = 32       # Shared IAM32 comparison; HiGAN+ explicitly native64→32.

OUTPUT_DIR = Path('/kaggle/working/ocr_benchmark_outputs')
DATA_DIR = Path('/kaggle/working/iam')
# Resume: attach prior Kaggle output and set the folder containing prepared/,
# datasets/ and ocr/. Full TrOCR checkpoint-N states resume automatically per arm.
# Keep data, seeds, counts and TROCR_CONFIG unchanged when continuing.
PREVIOUS_OUTPUT_DIR = None  # Output folder OR attached ocr_benchmark_resume.zip.keep file.
RESUME_ARCHIVE = OUTPUT_DIR.parent / "ocr_benchmark_resume.zip.keep"
CHECKPOINT_CACHE = Path('/tmp/ocr_benchmark_checkpoints')
CHECKPOINT_PATHS = {'spis_gan': None, 'higanplus': None, 'fw_gan': None}  # GAN weights only.
VALIDATION_PATH = None   # Otherwise hold out 10% of TRAIN writers, never use test.
TROCR_CONFIG = {
    'model_name': 'microsoft/trocr-small-stage1',
    'revision': 'd94c4048159050bb51e4105bad0a45dbfd5f3a02',
    'max_steps': 10000, 'micro_batch_size': 2, 'gradient_accumulation_steps': 16,
    'learning_rate': 5e-5, 'weight_decay': 0.01, 'max_grad_norm': 1.0,
    'warmup_ratio': 0.05, 'eval_steps': 500, 'save_steps': 500,
    'max_new_tokens': 64, 'num_beams': 1, 'fp16': False, 'bf16': False,
}
WANDB_MODE = 'auto'       # auto: online with key, otherwise disabled; or online/offline/disabled.
WANDB_PROJECT = 'iam-ocr-augmentation'

CODE_DIR = Path('/tmp/ocr_benchmark_code')
RUNTIME_DIR = Path('/tmp/ocr_benchmark_runtime')
SOURCES_DIR = Path('/tmp/ocr_benchmark_sources')
SESSION_HOURS = TIME_LIMIT
SESSION_STARTED = time.monotonic()
if not DATA_AUGMENT and not TRAIN_TROCR:
    raise ValueError('Enable DATA_AUGMENT or TRAIN_TROCR')
RUN_STAGE = 'all' if DATA_AUGMENT and TRAIN_TROCR else 'generation' if DATA_AUGMENT else 'ocr'
if (not MODELS or len(MODELS)!=len(set(MODELS)) or
        not set(MODELS) <= {'spis_gan','higanplus','fw_gan'}):
    raise ValueError('Select a nonempty unique list of supported generators')
if COHORT_HEIGHT not in (32,64) or (COHORT_HEIGHT==64 and MODELS!=['higanplus']):
    raise ValueError('Shared model comparison is IAM32; native64 supports only HiGAN+')
if not GPU_SLOTS or len(GPU_SLOTS)!=len(set(GPU_SLOTS)):
    raise ValueError('GPU_SLOTS must contain unique single GPU IDs')
if not 1 <= SYNTHETIC_COUNT <= 25000 or TIME_LIMIT <= 0:
    raise ValueError('Invalid sample count or session budget')
os.environ.update(HF_HOME='/tmp/ocr_benchmark_hf_cache', TOKENIZERS_PARALLELISM='false')
# Workers are standalone processes, not inline Jupyter plotting kernels.
WORKER_ENV = dict(os.environ, MPLBACKEND='Agg')
print('Stages:', RUN_STAGE, '| Models:', ', '.join(MODELS), '| real:', REAL_COUNT, '| synthetic/model:', SYNTHETIC_COUNT)


## 2. Install the benchmark runtime and connect W&B

This cell downloads the independent benchmark branch and installs its pinned
dependencies in a managed Python 3.11 environment, independently of Kaggle's kernel Python and ensurepip. It also checks the selected GPUs.
No GAN training branch is cloned, edited or trained. Secrets are not saved.


In [ ]:
# Isolated pinned runtime; model/source caches stay outside downloadable outputs.
def checked(argv, **kwargs):
    return subprocess.run([str(x) for x in argv], check=True, **kwargs)
if not CODE_DIR.exists():
    checked(['git','clone','--branch','ocr_benchmark','--single-branch',
             'https://github.com/rquq/HGGAN_test.git',CODE_DIR])
if not (CODE_DIR/'benchmark.py').is_file():
    raise RuntimeError('CODE_DIR is not the benchmark checkout')
# Refresh clean code before reading requirements; never reset local edits.
dirty = subprocess.check_output(['git','-C',str(CODE_DIR),'status','--porcelain'],text=True).strip()
if dirty: raise RuntimeError('Benchmark checkout has local edits; use a fresh CODE_DIR')
checked(['git','-C',CODE_DIR,'fetch','origin','ocr_benchmark'])
checked(['git','-C',CODE_DIR,'merge','--ff-only','origin/ocr_benchmark'])
code_head = subprocess.check_output(['git','-C',str(CODE_DIR),'rev-parse','HEAD'],text=True).strip()
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print('Benchmark source HEAD:', code_head)
requirements = CODE_DIR/'requirements.txt'
requirements_sha = hashlib.sha256(requirements.read_bytes()).hexdigest()
# The Kaggle kernel can be Python 3.13; workers use managed Python 3.11.
PYTHON = Path(subprocess.check_output(
    [sys.executable, str(CODE_DIR/'bootstrap_runtime.py'), '--runtime', str(RUNTIME_DIR)],
    text=True, env=WORKER_ENV).strip())
stamp = RUNTIME_DIR/'requirements.sha256'
if not stamp.exists() or stamp.read_text().strip() != requirements_sha:
    checked([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',requirements],env=WORKER_ENV)
    stamp.write_text(requirements_sha+'\n')
checked([PYTHON,'-c','import sys,torch,transformers,h5py,wandb,yaml,matplotlib.pyplot; print("Worker ready:",sys.version,torch.__version__)'],env=WORKER_ENV)
# Check GPU availability in a short worker, without loading models in the parent.
checked([PYTHON,'-c', 'import sys,torch; slots=sys.argv[1:]; assert all(s.isdigit() and int(s)<torch.cuda.device_count() for s in slots), "Requested GPU unavailable"', *GPU_SLOTS],env=WORKER_ENV)
if WANDB_MODE in ('online','auto') and not WORKER_ENV.get('WANDB_API_KEY'):
    try:
        from kaggle_secrets import UserSecretsClient
        WORKER_ENV['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_KEY')
    except Exception:
        if WANDB_MODE == 'online': raise
if WANDB_MODE == 'auto':
    WANDB_MODE = 'online' if WORKER_ENV.get('WANDB_API_KEY') else 'disabled'
if WANDB_MODE not in ('online','offline','disabled'): raise ValueError('Invalid WANDB_MODE')


In [ ]:
# Read-only resource overview; no model is loaded by this cell.
import shutil
usage = shutil.disk_usage(OUTPUT_DIR)
print(f'Writable output storage: {usage.free / 2**30:.1f} GiB free')
print('Outputs:', OUTPUT_DIR)
print('TrOCR state:', OUTPUT_DIR / 'ocr/<arm>/seed_<seed>/checkpoint-N')
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,memory.used',
                '--format=csv'], check=False)


## 3. Download or reuse IAM train/test data

Point DATA_DIR to attached data to avoid downloading it again. Otherwise the
selected IAM release files are downloaded and checked automatically. A real
validation partition is held out from training writers (or VALIDATION_PATH).
Test rows are never used as references, generation lexicon or OCR training data.


In [ ]:
# Real test is kept unchanged and used only for final recognizer evaluation.
config_path = CODE_DIR/'configs'/('benchmark_'+str(COHORT_HEIGHT)+'.yml')
raw_config = subprocess.check_output([str(PYTHON),'-c',
    'import yaml,json,sys; print(json.dumps(yaml.safe_load(open(sys.argv[1]))))',str(config_path)],text=True,env=WORKER_ENV)
runtime = json.loads(raw_config)
runtime['output_dir'] = str(OUTPUT_DIR.resolve())
runtime['wandb'] = {'mode':WANDB_MODE,'project':WANDB_PROJECT}
runtime['split_seed'] = SEED
runtime['dataset'].update(real_count=REAL_COUNT, validation=VALIDATION_PATH)
runtime['generation'].update(count=SYNTHETIC_COUNT, seed=SEED)
runtime['generation']['models'] = {name:runtime['generation']['models'][name] for name in MODELS}
runtime['ocr'] = dict(TROCR_CONFIG, seeds=[SEED])
runtime['dataset']['train'] = str((DATA_DIR/('iam_train_'+str(COHORT_HEIGHT)+'.hdf5')).resolve())
runtime['dataset']['test'] = str((DATA_DIR/('iam_test_'+str(COHORT_HEIGHT)+'.hdf5')).resolve())
DATA_DIR.mkdir(parents=True,exist_ok=True)
for split in ['train','test']:
    target = Path(runtime['dataset'][split])
    if not target.exists():
        url = 'https://github.com/rquq/HGGAN_test/releases/download/data/'+target.name
        fd, temporary = tempfile.mkstemp(prefix='.'+target.name,dir=target.parent)
        os.close(fd)
        try:
            urllib.request.urlretrieve(url,temporary)
            checked([PYTHON,'-c','import sys; from pipeline.data import validate_iam; validate_iam(sys.argv[1])',
                     temporary],cwd=CODE_DIR,env=WORKER_ENV)
            os.replace(temporary,target)
        finally:
            if os.path.exists(temporary): os.unlink(temporary)
    else:
        checked([PYTHON,'-c','import sys; from pipeline.data import validate_iam; validate_iam(sys.argv[1])',
                 target],cwd=CODE_DIR,env=WORKER_ENV)


## 4. Prepare frozen generator inference

The notebook fetches the pinned upstream sources and downloads the released GAN
weights with checksum verification. Existing verified downloads are reused. Only G and its required style encoder/backbone are constructed later.
With DATA_AUGMENT=False this step does not need GAN sources or weights.


In [ ]:
# G + required style encoders only. OCR-only resume requires NO GAN checkpoints.
model_configs = {}
source_records = {'benchmark_head':code_head,'requirements_sha256':requirements_sha,'generators':{}}
for name, relative in runtime['generation']['models'].items():
    raw = subprocess.check_output([str(PYTHON),'-c',
        'import yaml,json,sys; print(json.dumps(yaml.safe_load(open(sys.argv[1]))))',str(CODE_DIR/relative)],text=True,env=WORKER_ENV)
    model = json.loads(raw)
    if DATA_AUGMENT:
        checkout = SOURCES_DIR/name
        checkout.parent.mkdir(parents=True,exist_ok=True)
        if not checkout.exists():
            checked(['git','clone',model['source_url'],checkout])
            checked(['git','-C',checkout,'checkout','--detach',model['source_revision']])
        head = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip()
        if head != model['source_revision']: raise RuntimeError('Source revision mismatch: '+name)
        source_root = checkout/'HiGAN+' if name=='higanplus' else checkout
        model['source_root'] = str(source_root)
        model['source_config'] = str(source_root/'configs'/Path(model['source_config']).name)
        explicit = CHECKPOINT_PATHS.get(name)
        if explicit:
            checkpoint = Path(explicit)
        else:
            downloaded = subprocess.check_output(
                [str(PYTHON), '-m', 'pipeline.assets', '--model', name,
                 '--cache', str(CHECKPOINT_CACHE)], cwd=CODE_DIR, env=WORKER_ENV, text=True)
            checkpoint = Path(downloaded.strip())
        if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)
        model['checkpoint'] = str(checkpoint.resolve())
        source_records['generators'][name] = {'revision':head,'url':model['source_url']}
    model_path = OUTPUT_DIR/('runtime_'+name+'.json')
    model_path.write_text(json.dumps(model,indent=2))
    model_configs[name] = str(model_path)
runtime['generation']['models'] = model_configs
RUNTIME_CONFIG = OUTPUT_DIR/'runtime_config.json'
RUNTIME_CONFIG.write_text(json.dumps(runtime,indent=2))
(OUTPUT_DIR/'source_manifest.json').write_text(json.dumps(source_records,indent=2))
# Validate the actual serialized recipe before any generation or model load.
checked([PYTHON, '-c',
    'import sys; from pipeline.config import load_config; c=load_config(sys.argv[1]); print("Validated OCR learning rate:",c["ocr"]["learning_rate"],type(c["ocr"]["learning_rate"]).__name__)',
    RUNTIME_CONFIG], cwd=CODE_DIR, env=WORKER_ENV)


## 5. Restore prior outputs and freeze the shared cohort

Fresh run: select the same X training rows and validation data for all arms, then
freeze one reference/text/seed manifest. Synthetic targets differ from their own
reference transcripts. Resume: restore prepared data, generation shards, datasets
and full per-arm TrOCR checkpoints into the writable output folder. Conflicting
data/recipe identities are rejected rather than silently starting a different run.


In [ ]:
# Restore the full prior output tree into writable storage, then verify identities.
# Includes each arm's TrOCR checkpoint-N, optimizer, scheduler, RNG, best and history.
base_argv = [str(PYTHON),str(CODE_DIR/'benchmark.py')]
prepare_argv = base_argv+['prepare','--config',str(RUNTIME_CONFIG)]
if PREVIOUS_OUTPUT_DIR is not None:
    prepare_argv += ['--previous-output-dir',str(PREVIOUS_OUTPUT_DIR)]
checked(prepare_argv,cwd=CODE_DIR,env=WORKER_ENV)


In [ ]:
# Read-only overview. File counts are not proof of checkpoint validity;
# the pipeline validates identities and full native training state before resume.
checkpoint_rows = []
print('SAVED OCR JOBS: arm | status | recorded update | checkpoint folders | output')
for arm in ['real_only', *MODELS]:
    directory = OUTPUT_DIR / 'ocr' / arm / f'seed_{SEED}'
    receipt = {}
    if (directory / 'status.json').is_file():
        try:
            receipt = json.loads((directory / 'status.json').read_text())
            if not isinstance(receipt, dict): receipt = {'status': 'unreadable'}
        except (OSError, ValueError):
            receipt = {'status': 'unreadable'}
    checkpoints = [p for p in directory.glob('checkpoint-*') if p.is_dir()]
    row = dict(arm=arm, status=receipt.get('status', 'not_started'),
               global_step=receipt.get('global_step', 0), saved_checkpoints=len(checkpoints),
               directory=str(directory))
    checkpoint_rows.append(row)
    print(' | '.join(str(row[k]) for k in ('arm','status','global_step','saved_checkpoints','directory')))


## 6. Generate matched datasets → train TrOCR → final test

Run this cell and leave it running. It streams worker logs and stops when the
jobs finish or cooperatively pause for the session deadline. No separate monitor
cell or printed-keyword detector is needed. Generation must finish successfully
before OCR training starts. A failed/incomplete export is never used as training
data. Two independent jobs use the two T4s; the real-only control is also queued.


In [ ]:
# One command: finish/resume BOTH datasets, then train real-only and augmented OCRs.
# Complete datasets/jobs are verified and skipped, not regenerated/retrained.
# The subprocess queue streams logs and exits when jobs finish or safely pause.
remaining = max(0.,TIME_LIMIT*3600-(time.monotonic()-SESSION_STARTED))
try:
    checked(base_argv+['run','--config',str(RUNTIME_CONFIG),'--stage',RUN_STAGE,
                      '--gpu-slots',','.join(GPU_SLOTS),'--time-budget-seconds',str(remaining)],
            cwd=CODE_DIR,env=WORKER_ENV)
finally:
    # Preserve all nested identities/checkpoints in ONE upload file, even on failure.
    bundle = subprocess.run([str(PYTHON), '-m', 'pipeline.bundle', '--output', str(OUTPUT_DIR),
                             '--archive', str(RESUME_ARCHIVE)], cwd=CODE_DIR, env=WORKER_ENV)
    if bundle.returncode:
        print('Resume archive creation failed; preserve the full OUTPUT_DIR instead.')


## 7. Inspect results and generated-data previews

Final CER/WER come from separately trained TrOCR checkpoints selected on the same
real validation data, evaluated on the complete real IAM test set. Paused or
incompatible runs have no final score. Previews below are only for inspection;
they are never used to filter samples or select a model.


In [ ]:
# Scores appear only after a complete final IAM test from validation-selected best.
checked(base_argv+['report','--config',str(RUNTIME_CONFIG)],cwd=CODE_DIR,env=WORKER_ENV)
import csv
with (OUTPUT_DIR/'results.csv').open() as stream:
    results = list(csv.DictReader(stream))
print('RESULTS: arm | status | update | CER% | WER% | word accuracy%')
for row in results:
    print(' | '.join(str(row.get(k,'—') or '—') for k in
          ('arm','status','global_step','cer_percent','wer_percent','word_accuracy_percent')))


In [ ]:
# Existing export previews: reference/text beside target/generated image.
from IPython.display import Image, display
for name in MODELS:
    preview = OUTPUT_DIR / 'datasets' / name / 'preview.png'
    print(name + ':')
    if preview.is_file():
        display(Image(filename=str(preview)))
    else:
        print('No completed preview yet; inspect generation status/logs.')


## 8. Save datasets, reports and full continuation state

Each dataset ZIP is downloadable on its own. Kaggle Save Version also keeps the
full OUTPUT_DIR, including separate native TrOCR training checkpoint folders.
Keep that full folder for the next session, not only the dataset ZIPs or best
weights. Environment/source/Hugging Face caches stay under /tmp, not in outputs.


In [ ]:
# Keep the complete output folder for resume; ZIPs below are dataset exports only.
from IPython.display import FileLink, display
download_paths = ([RESUME_ARCHIVE] if RESUME_ARCHIVE.is_file() else [])
download_paths += sorted(OUTPUT_DIR.glob('datasets/*/dataset.zip'))
download_paths += [p for p in (OUTPUT_DIR/'prepared.zip', OUTPUT_DIR/'results.csv',
                             OUTPUT_DIR/'results.json', OUTPUT_DIR/'runtime_config.json') if p.is_file()]
for path in download_paths:
    print(path)
    display(FileLink(str(path)))
print('FULL RESUME OUTPUT:', OUTPUT_DIR)
print('TrOCR state: ocr/<arm>/seed_<seed>/checkpoint-N/ (optimizer + scheduler + RNG included)')
print('Next session: attach this Output; set PREVIOUS_OUTPUT_DIR to its ocr_benchmark_outputs folder.')
print('Upload only ocr_benchmark_resume.zip.keep as ONE Kaggle input file; do not select nested files.')
print('Set PREVIOUS_OUTPUT_DIR to that attached archive file; restoration is automatic.')


## Saved output and continuation

```
ocr_benchmark_outputs/
  prepared/                    # shared real train/validation + paired manifest
  datasets/spis_gan/            # synthetic.hdf5, preview, metadata, dataset.zip
  datasets/higanplus/           # separate synthetic dataset, same manifest
  ocr/real_only/seed_123456/     # control TrOCR checkpoints / history / metrics
  ocr/spis_gan/seed_123456/      # real + SpiS-GAN TrOCR checkpoints
  ocr/higanplus/seed_123456/     # real + HiGAN+ TrOCR checkpoints
  logs/                        # generation and TrOCR job logs / receipts
  results.csv
```

**Continue next Kaggle session:** attach the prior Output; set
`PREVIOUS_OUTPUT_DIR` to the folder containing `prepared`, `datasets`, `ocr`.
Keep counts, seeds and TrOCR recipe unchanged. Keep **full checkpoint-N folders**
and selection metadata, not only model.safetensors: native resume retains model,
optimizer, scheduler, RNG and data position. The latest complete native checkpoint
is used; the validation-selected best is preserved separately. Different arms
never share trained weights. Existing completed jobs are verified and skipped.

With both toggles true, completed generation is reused before continuing OCR.
Set `DATA_AUGMENT=False, TRAIN_TROCR=True` for OCR-only continuation from saved
datasets: no GAN sources/checkpoints are needed. Generation-only is True/False.

Training uses identical pinned **TrOCR-small-stage1** initialization, not a frozen
handwriting recognizer. FP32, effective batch32, real-validation CER/WER checkpoint
selection and final full real IAM test. No generated-image filtering and no test
normalization/exclusion. Dataset ZIPs can be downloaded independently; **retain
the full output folder including ocr/ for training continuation**.

The session budget is cooperative: an in-flight save/update/model load can exceed
the boundary, so leave margin. Paused jobs are not final results; rerun from the
saved outputs. One 2×T4 session is not promised to finish all three OCR arms.
Upstream checkpoint training/test exposure remains unknown where undocumented;
this is a controlled downstream augmentation comparison, not a claim of exact
paper recipe reproduction. Full-size Kaggle performance must be measured there.

### Avoid duplicate-filename upload errors

`identity.json`, `status.json` and `metadata.json` legitimately repeat in different
directories. Never flatten or rename those files. The run cell now creates
`/kaggle/working/ocr_benchmark_resume.zip.keep` after completion or a Python-level
failure. The `.keep` suffix is intentional: keep this as one opaque upload file,
not a set of extracted files in the dataset uploader. Attach that one file and
set PREVIOUS_OUTPUT_DIR to its Kaggle input path. The pipeline recognizes its
ZIP contents, validates paths and restores the original directory structure.
Canonical checkpoint-N folders are included; redundant best/last directory
aliases are omitted and rebuilt from native state during restore.
